<div style="background:linear-gradient(120deg,#0b4a43 0%,#0e7a6e 55%,#0f6cbd 100%);background-color:#0e7a6e;border-radius:12px;padding:22px 28px;color:#ffffff"><table style="border:none;border-collapse:collapse;width:100%;margin:0"><tr style="border:none;background:none"><td style="border:none;padding:0;vertical-align:middle;text-align:left;color:#ffffff"><div style="font-size:13px;letter-spacing:2px;text-transform:uppercase;opacity:0.85">Microsoft Fabric &middot; Lakehouse &middot; Component</div><div style="font-size:34px;font-weight:700;line-height:1.2;margin:2px 0 6px 0">Restore Form</div><div style="font-size:15px;line-height:1.5;max-width:520px">The input screen for a restore: pick the backup by browsing the write-once container, name the target lakehouse, and choose what happens if the target is not empty. It checks the inputs and looks up the IDs.</div></td><td style="border:none;padding:0 0 0 20px;vertical-align:middle;text-align:right"></td></tr></table></div><div style="margin:12px 2px 0 2px;font-size:14px;line-height:1.6"><b>This notebook is not opened to do a job.</b> It is loaded by the Restore Utility notebook, which shows the form and then uses what was entered.<br/><span style="display:inline-block;background:#107c10;color:#ffffff;border-radius:11px;padding:1px 10px;font-size:12px;font-weight:600">Reads only</span> &nbsp; <span style="display:inline-block;background:#107c10;color:#ffffff;border-radius:11px;padding:1px 10px;font-size:12px;font-weight:600">Copies and deletes nothing</span> &nbsp; <span style="display:inline-block;background:#5c2e91;color:#ffffff;border-radius:11px;padding:1px 10px;font-size:12px;font-weight:600">Edit only to change the form</span></div>


<div style="border-left:6px solid #5c2e91;border-radius:6px;padding:10px 16px;margin:14px 0 4px 0;background:rgba(127,127,127,0.09)"><table style="border:none;border-collapse:collapse;margin:0"><tr style="border:none;background:none"><td style="border:none;padding:0 14px 0 0;vertical-align:top"></td><td style="border:none;padding:0;vertical-align:top;text-align:left"><div style="font-size:12px;letter-spacing:1.5px;text-transform:uppercase;color:#5c2e91;font-weight:700">Part 1</div><div style="font-size:20px;font-weight:600;margin:1px 0 6px 0">Form code</div><div style="font-size:14px;line-height:1.55">Defines everything the form needs. Nothing runs until the form is shown in Part 2.<ul style="margin:6px 0 6px 0"><li><b>Lookups</b> (<code>fabric_list</code>, <code>find_by_name</code>): find a workspace or lakehouse by name, as the signed-in person, and return its ID.</li><li><b>Container browsing</b> (<code>browse_backup_folders</code>, <code>list_backup_timestamps</code>): list the folders that hold backups and the complete backups in a folder, newest first.</li><li><b>Checking</b> (<code>resolve_restore_inputs</code>): confirms the backup is complete, finds the target IDs, and insists on the typed lakehouse name before any delete option is accepted.</li><li><b>The form</b> (<code>RestoreInputForm</code>): the boxes, lists and two buttons, <b>Browse</b> and <b>Check inputs</b>.</li></ul><b>Expect:</b> no output. <b>Result for later cells:</b> after <b>Check inputs</b> succeeds, the values are in <code>restore_form.params</code>; any change on the form clears them until the button is pressed again.</div></td></tr></table></div>


In [ ]:
import html, re, time
import requests
import ipywidgets as W
import notebookutils

FABRIC_API = "https://api.fabric.microsoft.com/v1/"


def fabric_list(path):
    """All pages of a Fabric REST list call, as the signed-in user."""
    headers = {"Authorization": "Bearer " + notebookutils.credentials.getToken("pbi")}
    items, token = [], None
    while True:
        for attempt in range(1, 6):
            response = requests.get(FABRIC_API + path, headers=headers, timeout=60,
                                    params={"continuationToken": token} if token else None)
            if response.status_code in (429, 500, 502, 503, 504) and attempt < 5:
                time.sleep(int(response.headers.get("Retry-After", 2 ** attempt)))
                continue
            break
        response.raise_for_status()
        page = response.json()
        items += page.get("value", [])
        token = page.get("continuationToken")
        if not token:
            return items


def find_by_name(items, name, kind, where=""):
    """The one item whose displayName equals `name` (upper/lower case ignored)."""
    name = name.strip()
    if not name:
        raise ValueError(f"{kind}: nothing was typed.")
    hits = [i for i in items if i["displayName"].lower() == name.lower()]
    if len(hits) == 1:
        return hits[0]
    if len(hits) > 1:
        raise ValueError(f'{kind}: {len(hits)} items are named "{name}"{where}. Names must be unique.')
    close = sorted(i["displayName"] for i in items if name.lower() in i["displayName"].lower())[:5]
    hint = f" Did you mean: {', '.join(close)}?" if close else ""
    raise ValueError(f'{kind}: "{name}" was not found{where}, or you have no access to it.{hint}')


def container_address(text):
    """Accepts abfss://container@account.dfs.core.windows.net[/folder]
    or https://account.dfs.core.windows.net/container[/folder]. Returns the abfss form."""
    text = text.strip().rstrip("/")
    if not text:
        raise ValueError("Backup location: nothing was typed.")
    m = re.fullmatch(r"https://([^./]+)\.(?:dfs|blob)\.core\.windows\.net/([^/]+)(/.*)?", text)
    if m:
        text = f"abfss://{m.group(2)}@{m.group(1)}.dfs.core.windows.net{m.group(3) or ''}"
    if not re.fullmatch(r"abfss://[^@/]+@[^./]+\.dfs\.core\.windows\.net(/.*)?", text):
        raise ValueError("Backup location must look like "
                         "abfss://<container>@<account>.dfs.core.windows.net/<folder>")
    return text


def name_set(text):
    """'dbo, sales' -> {'dbo', 'sales'}. Blank -> None, which means everything."""
    names = {part.strip() for part in str(text).split(",") if part.strip()}
    return names or None


def browse_backup_folders(container):
    """Looks in the typed address for folders that hold backups.
    Returns (address, folders); the folder '' means the address itself holds backups."""
    address = container_address(container)
    try:
        items = notebookutils.fs.ls(address)
    except Exception as exc:
        raise ValueError(f"Could not read {address}. Check the address and your access. "
                         + " ".join(str(exc).split())[:200])
    names = sorted(i.name.rstrip("/") for i in items if i.isDir)
    folders = [""] if "backups" in names else []
    folders += [n for n in names if n != "backups" and notebookutils.fs.exists(f"{address}/{n}/backups")]
    return address, folders


def list_backup_timestamps(archive):
    """(complete backups newest first, number of incomplete runs) under <archive>/backups."""
    folders = [i.name.rstrip("/") for i in notebookutils.fs.ls(f"{archive}/backups") if i.isDir]
    complete = sorted((ts for ts in folders
                       if notebookutils.fs.exists(f"{archive}/backups/{ts}/_manifest/manifest.json")),
                      reverse=True)
    return complete, len(folders) - len(complete)


def timestamp_label(ts):
    return f"{ts[:4]}-{ts[4:6]}-{ts[6:8]} {ts[8:10]}:{ts[10:12]} UTC   ({ts})"


def resolve_restore_inputs(backup_location, backup_timestamp, target_workspace, target_lakehouse,
                           restore_files, only_schemas, only_tables, clean_target, confirm_target_name):
    """Turns the typed values into what the restore needs. Reads only; changes nothing."""
    archive = container_address(backup_location)
    try:
        folders = [i.name.rstrip("/") for i in notebookutils.fs.ls(f"{archive}/backups") if i.isDir]
    except Exception as exc:
        raise ValueError(f"No backups could be read under {archive}/backups. Check the address, including "
                         "the folder, and your access. " + " ".join(str(exc).split())[:200])
    complete = sorted(ts for ts in folders
                      if notebookutils.fs.exists(f"{archive}/backups/{ts}/_manifest/manifest.json"))
    if not complete:
        raise ValueError(f"There is no complete backup under {archive}/backups.")
    timestamp = str(backup_timestamp).strip()
    if not timestamp:
        timestamp = complete[-1]                                  # blank = latest complete backup
    elif timestamp not in complete:
        state = "is incomplete" if timestamp in folders else "does not exist"
        raise ValueError(f"Backup {timestamp} {state}. Complete backups: {', '.join(reversed(complete[-10:]))}")

    workspace = find_by_name(fabric_list("workspaces"), target_workspace, "Target workspace")
    lakehouse = find_by_name(fabric_list(f"workspaces/{workspace['id']}/lakehouses"), target_lakehouse,
                             "Target lakehouse", f' in workspace "{workspace["displayName"]}"')

    confirm = str(confirm_target_name).strip()
    if clean_target and confirm != lakehouse["displayName"]:
        raise ValueError("Deleting from the target was chosen. To allow it, type the target lakehouse name "
                         f'exactly in "Confirm lakehouse name": {lakehouse["displayName"]}')
    return {
        "archive": archive,
        "backup_timestamp": timestamp,
        "complete_backups": list(reversed(complete)),
        "target_workspace_name": workspace["displayName"], "target_workspace_id": workspace["id"],
        "target_lakehouse_name": lakehouse["displayName"], "target_lakehouse_id": lakehouse["id"],
        "restore_files": bool(restore_files),
        "only_schemas": name_set(only_schemas),
        "only_tables": name_set(only_tables),
        "clean_target": clean_target,
        "confirm_target_name": confirm if clean_target else None,
    }


class RestoreInputForm:
    """Captures every input a restore needs. After a successful press of the button,
    the values are in `.params`. The form itself changes nothing."""

    def __init__(self):
        wide, style = W.Layout(width="760px"), {"description_width": "200px"}
        def box(label, hint):
            return W.Text(description=label, placeholder=hint, layout=wide, style=style)
        self.container = W.Text(description="Backup container", style=style, layout=W.Layout(width="620px"),
                                placeholder="abfss://<container>@<account>.dfs.core.windows.net")
        self.browse_button = W.Button(description="Browse", layout=W.Layout(width="132px"))
        self.backup_folder = W.Dropdown(description="Backup folder", options=[], layout=wide, style=style)
        self.backup_timestamp = W.Dropdown(description="Backup (date and time)", options=[], layout=wide, style=style)
        self.note = W.HTML()
        self._address = None                    # the container address the folder list was read from
        self._loading = False
        self.target_workspace = box("Target workspace name", "workspace that holds the lakehouse to restore into")
        self.target_lakehouse = box("Target lakehouse name", "lakehouse to restore into")
        self.restore_files = W.Checkbox(value=True, description="Also restore everything under Files",
                                        indent=False, layout=wide)
        self.only_schemas = box("Only these schemas", "leave blank for all, or e.g. dbo")
        self.only_tables = box("Only these tables", "leave blank for all, or e.g. account, contact")
        self.clean_target = W.Dropdown(description="If the target is not empty", layout=wide, style=style, options=[
            ("Stop and change nothing (safe)", False),
            ("Delete only what the backup puts back, then restore", "replace"),
            ("Delete EVERY table and file in the target, then restore", "all")])
        self.confirm_target_name = box("Confirm lakehouse name", "only when deleting: type the target lakehouse name")
        self.button = W.Button(description="Check inputs", button_style="info", layout=W.Layout(width="200px"))
        self.result = W.HTML()
        self.params = None
        self.button.on_click(lambda _b: self.submit())
        self.browse_button.on_click(lambda _b: self.browse())
        self.container.observe(self._container_edited, names="value")
        self.backup_folder.observe(self._folder_chosen, names="value")
        self._inputs = [self.backup_timestamp, self.target_workspace, self.target_lakehouse,
                        self.restore_files, self.only_schemas, self.only_tables, self.clean_target,
                        self.confirm_target_name]
        for w in self._inputs:                  # any change after the check clears the result
            w.observe(self._changed, names="value")
        self.box = W.VBox([W.HBox([self.container, self.browse_button]), self.backup_folder,
                           self.backup_timestamp, self.note] + self._inputs[1:] + [self.button, self.result])

    def show(self):
        display(self.box)

    def _say(self, colour, body):
        self.result.value = (f'<div style="border-left:5px solid {colour};padding:8px 12px;'
                             f'margin:8px 0;font-size:14px">{body}</div>')

    def _changed(self, _change):
        if self._loading:
            return
        if self.params is not None:
            self.params = None
            self._say("#0078d4", "A value changed. Press <b>Check inputs</b> again.")

    def _set_options(self, dropdown, options):
        self._loading = True
        try:
            dropdown.options = options
            dropdown.value = options[0][1] if options else None
        finally:
            self._loading = False

    def _container_edited(self, _change):
        """The folder and backup lists belong to the address that was browsed; clear them when it changes."""
        self._address = None
        self._set_options(self.backup_folder, [])
        self._set_options(self.backup_timestamp, [])
        self.note.value = ""
        self._changed(None)

    def _archive(self):
        if self._address is None or self.backup_folder.value is None:
            return None
        return self._address + ("/" + self.backup_folder.value if self.backup_folder.value else "")

    def browse(self):
        """What Browse does: fills the folder list, then the backup list. Reads only.
        Can also be called from a cell: restore_form.browse()"""
        self.params = None
        self.browse_button.disabled = True
        self._say("#0078d4", "Reading the container ...")
        try:
            address, folders = browse_backup_folders(self.container.value)
            if not folders:
                raise ValueError(f"No folder with backups was found in {address}.")
            self._address = address
            self._set_options(self.backup_folder, [(f or "(this location)", f) for f in folders])
            self._load_timestamps()
            self._say("#0078d4", f"Found {len(folders)} folder(s) with backups. Choose the folder and the backup, "
                                 "fill in the rest, then press <b>Check inputs</b>.")
        except Exception as exc:
            self._address = None
            self._set_options(self.backup_folder, [])
            self._set_options(self.backup_timestamp, [])
            self.note.value = ""
            self._say("#d13438", "<b>Could not browse.</b> " + html.escape(str(exc)))
            print("COULD NOT BROWSE:", exc)
        finally:
            self.browse_button.disabled = False

    def _load_timestamps(self):
        complete, incomplete = list_backup_timestamps(self._archive())
        self._set_options(self.backup_timestamp, [(timestamp_label(ts), ts) for ts in complete])
        self.note.value = (f'<span style="font-size:12px">{len(complete)} complete backup(s), newest first.'
                           + (f" {incomplete} incomplete run(s) not listed." if incomplete else "") + "</span>")

    def _folder_chosen(self, _change):
        if self._loading:
            return
        try:
            self._load_timestamps()
        except Exception as exc:
            self._set_options(self.backup_timestamp, [])
            self._say("#d13438", "<b>Could not list the backups.</b> " + html.escape(str(exc)))
        self._changed(None)

    def submit(self):
        """What the button does. Can also be called from a cell: restore_form.submit()"""
        self.params = None
        self.button.disabled = True
        self._say("#0078d4", "Checking ...")
        try:
            if self._archive() is None or not self.backup_timestamp.value:
                raise ValueError("Type the backup container, press Browse, then choose the folder and the backup.")
            p = resolve_restore_inputs(
                self._archive(), self.backup_timestamp.value,
                self.target_workspace.value, self.target_lakehouse.value,
                self.restore_files.value, self.only_schemas.value, self.only_tables.value,
                self.clean_target.value, self.confirm_target_name.value)
        except Exception as exc:
            self._say("#d13438", "<b>Not accepted.</b> " + html.escape(str(exc)))
            print("NOT ACCEPTED:", exc)
            return None
        finally:
            self.button.disabled = False
        self.params = p
        ts = p["backup_timestamp"]
        when = f"{ts[:4]}-{ts[4:6]}-{ts[6:8]} {ts[8:10]}:{ts[10:12]} UTC"
        latest = " (the latest)" if ts == p["complete_backups"][0] else ""
        cleaning = {False: "nothing is deleted; the restore stops if anything is in the way",
                    "replace": "DELETE what the backup puts back, then restore",
                    "all": "DELETE EVERY table and file in the target, then restore"}[p["clean_target"]]
        rows = [("Backup location", p["archive"], ""),
                ("Backup", ts + latest, when),
                ("Target workspace", p["target_workspace_name"], p["target_workspace_id"]),
                ("Target lakehouse", p["target_lakehouse_name"], p["target_lakehouse_id"]),
                ("Restore Files", "yes" if p["restore_files"] else "no", ""),
                ("Schemas", ", ".join(sorted(p["only_schemas"])) if p["only_schemas"] else "all", ""),
                ("Tables", ", ".join(sorted(p["only_tables"])) if p["only_tables"] else "all", ""),
                ("If target is not empty", cleaning, "")]
        cells = "".join("<tr>" + "".join(f'<td style="padding:3px 14px 3px 0">{html.escape(str(c))}</td>'
                                         for c in r) + "</tr>" for r in rows)
        self._say("#c19c00" if p["clean_target"] else "#107c10",
                  "<b>Inputs accepted.</b> Nothing has been changed. Run the preview cell next."
                  + f'<table style="margin-top:6px;font-size:13px">{cells}</table>'
                  + f'<div style="font-size:12px;margin-top:4px">Complete backups here, newest first: '
                  + html.escape(", ".join(p["complete_backups"][:10])) + "</div>")
        for key, value in p.items():
            print(f"{key:24} = {value}")
        return p


<div style="border-left:6px solid #0f6cbd;border-radius:6px;padding:10px 16px;margin:14px 0 4px 0;background:rgba(127,127,127,0.09)"><table style="border:none;border-collapse:collapse;margin:0"><tr style="border:none;background:none"><td style="border:none;padding:0 14px 0 0;vertical-align:top"></td><td style="border:none;padding:0;vertical-align:top;text-align:left"><div style="font-size:12px;letter-spacing:1.5px;text-transform:uppercase;color:#0f6cbd;font-weight:700">Part 2</div><div style="font-size:20px;font-weight:600;margin:1px 0 6px 0">Show the form</div><div style="font-size:14px;line-height:1.55">Creates the form and displays it. This is the cell whose output the person works in.<br/><b>To use it:</b> type the container address, press <b>Browse</b>, pick the folder and the backup, name the target workspace and lakehouse, then press <b>Check inputs</b>.<br/><b>Expect:</b> a line <i>Inputs accepted</i> with the backup, the target and its IDs, or a red line saying what is wrong.<br/><b>If a button does nothing:</b> run <code>restore_form.browse()</code> or <code>restore_form.submit()</code> in a cell; they do the same as the buttons.</div></td></tr></table></div>


In [ ]:
restore_form = RestoreInputForm()
restore_form.show()